# THESIS-009 — Experiment 2C: Spike Observation (SQ3)
**Story Points:** 2 | **Status:** TODO (depends on THESIS-007)

Run L6 (10 concurrent) on K8s. Observe scheduling delays, pending pods, and failures.
Answers SQ3 at extreme load.


## Acceptance Criteria
- [ ] 3 reps at L6 (10 concurrent) on K8s
- [ ] Pending pods over time recorded
- [ ] Time-to-all-pods-running recorded
- [ ] Any scheduling timeouts or failures noted
- [ ] `pod_timing.csv` per run with scheduling latency at extreme load

## Launch spike experiment

In [ ]:
import subprocess
print('To run:')
print('  cd .. && DAGSTER_PORT=3001 bash scripts/run_experiment.sh exp2c k8s --levels 10')
print('  # or: make exp2c-spike')
# Uncomment to execute:
# r = subprocess.run(
#     ['bash', '../scripts/run_experiment.sh', 'exp2c', 'k8s', '--levels', '10'],
#     text=True, cwd='..'
# )

## Monitor pending pods during L6

In [ ]:
import subprocess, time
print('Watching pods in dagster namespace (Ctrl-C to stop):')
try:
    for i in range(10):
        r = subprocess.run(
            ['kubectl', 'get', 'pods', '-n', 'dagster', '--no-headers'],
            capture_output=True, text=True
        )
        lines = r.stdout.strip().splitlines()
        pending = sum(1 for l in lines if 'Pending' in l)
        running = sum(1 for l in lines if 'Running' in l)
        total   = len(lines)
        print(f'  t+{i*5:3d}s: {running} Running, {pending} Pending / {total} total')
        time.sleep(5)
except KeyboardInterrupt:
    pass

## Analyse pod timing at L6

In [ ]:
import pandas as pd
from pathlib import Path
exp_dir = Path('../data/raw/exp2-kubernetes-isolation/part-a')
timing_csvs = list(exp_dir.rglob('pod_timing.csv'))
if not timing_csvs:
    print('No pod_timing.csv found yet — run experiment first')
else:
    dfs = [pd.read_csv(c) for c in timing_csvs]
    combined = pd.concat(dfs, ignore_index=True)
    if 'submitted_ts' in combined.columns and 'running_ts' in combined.columns:
        combined['submitted_ts'] = pd.to_datetime(combined['submitted_ts'], utc=True, errors='coerce')
        combined['running_ts']   = pd.to_datetime(combined['running_ts'], utc=True, errors='coerce')
        combined['sched_s'] = (combined['running_ts'] - combined['submitted_ts']).dt.total_seconds()
        print(combined.groupby('level')['sched_s'].describe().round(2))
    else:
        print(combined.head())